In [ ]:
import os
import re
import pandas as pd
import subprocess
from tqdm import tqdm
import csv
import shutil

# Function to slow down a video
def slow_down_video(input_path, output_path, speed_factor=0.5):
    """
    Slow down a video by the specified factor using ffmpeg.
    
    Args:
        input_path: Path to the input video
        output_path: Path to save the output video
        speed_factor: Factor to slow down the video (e.g., 0.5 for half speed)
    """
    # Create output directory if it doesn't exist
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    # Build ffmpeg command: use setpts filter to slow down video, atempo to slow down audio
    # atempo filter only supports 0.5-2.0 range, so for extreme slowing, we chain it
    cmd = [
        'ffmpeg', 
        '-i', input_path, 
        '-filter_complex', 
        f'[0:v]setpts={1/speed_factor}*PTS[v];[0:a]atempo={speed_factor}[a]', 
        '-map', '[v]', 
        '-map', '[a]', 
        '-c:v', 'libx264', 
        '-preset', 'medium', 
        '-c:a', 'aac', 
        output_path,
        '-y'  # Overwrite output file if it exists
    ]
    
    subprocess.run(cmd, check=True)

# Function to adjust SRT timestamps
def adjust_srt_timestamps(srt_text, speed_factor=0.5):
    """
    Adjust SRT timestamps by the specified speed factor.
    
    Args:
        srt_text: SRT format text
        speed_factor: Factor to slow down the timestamps (e.g., 0.5 for double duration)
    
    Returns:
        Adjusted SRT text
    """
    # Define a regex pattern to match SRT timestamp lines
    pattern = r'(\d+:\d+:\d+,\d+) --> (\d+:\d+:\d+,\d+)'
    
    def time_to_seconds(time_str):
        """Convert SRT timestamp to seconds"""
        h, m, s = time_str.replace(',', '.').split(':')
        return float(h) * 3600 + float(m) * 60 + float(s)
    
    def seconds_to_time(seconds):
        """Convert seconds to SRT timestamp format"""
        h = int(seconds // 3600)
        m = int((seconds % 3600) // 60)
        s = seconds % 60
        ms = int((s - int(s)) * 1000)
        s = int(s)
        return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"
    
    def adjust_timestamp(match):
        """Adjust a pair of timestamps based on the speed factor"""
        start_time = match.group(1)
        end_time = match.group(2)
        
        # Convert to seconds
        start_seconds = time_to_seconds(start_time)
        end_seconds = time_to_seconds(end_time)
        
        # Adjust time by dividing by speed factor (e.g., 0.5 makes duration 2x longer)
        adjusted_start = start_seconds / speed_factor
        adjusted_end = end_seconds / speed_factor
        
        # Convert back to timestamp format
        adjusted_start_time = seconds_to_time(adjusted_start)
        adjusted_end_time = seconds_to_time(adjusted_end)
        
        return f"{adjusted_start_time} --> {adjusted_end_time}"
    
    # Use regex to replace all timestamp lines
    adjusted_srt = re.sub(pattern, adjust_timestamp, srt_text)
    return adjusted_srt

# Main execution

# Create output folder
output_folder = "videos_full_slow"
os.makedirs(output_folder, exist_ok=True)

# Process metadata file
metadata_file = "video_metadata.csv"
output_metadata_file = "videos_full_slow_metadata.csv"

print("Reading metadata file...")
df = pd.read_csv(metadata_file)

# Create copies of dataframe columns to modify
df['slowed_video_path'] = df['video_path'].copy()
df['slowed_caption'] = df['caption'].copy()

# Process each video and update metadata
print("Processing videos and updating metadata...")
for idx, row in tqdm(df.iterrows(), total=len(df)):
    # Get video info
    video_path = row['video_path']
    
    # Check if video exists
    if not os.path.exists(video_path):
        print(f"Warning: Video not found: {video_path}")
        continue
    
    # Create output path
    output_path = video_path.replace("videos/", "videos_full_slow/")
    
    # Slow down the video
    try:
        slow_down_video(video_path, output_path)
        
        # Update metadata
        df.at[idx, 'slowed_video_path'] = output_path
        
        # Update caption timestamps if available
        if pd.notna(row['caption']):
            adjusted_caption = adjust_srt_timestamps(row['caption'])
            df.at[idx, 'slowed_caption'] = adjusted_caption
    
    except Exception as e:
        print(f"Error processing {video_path}: {str(e)}")

# Create final output dataframe
output_df = df.copy()
output_df['video_path'] = df['slowed_video_path']  # Replace with slowed path
output_df['caption'] = df['slowed_caption']  # Replace with adjusted captions
output_df = output_df.drop(['slowed_video_path', 'slowed_caption'], axis=1)  # Remove temporary columns
# Double the duration of the video
output_df['duration'] = output_df['duration'] * 2

# Save updated metadata to CSV
print(f"Saving updated metadata to {output_metadata_file}...")
output_df.to_csv(output_metadata_file, index=False)

print("Processing complete!")

# Show a summary of the processing
num_videos = len(df)
num_processed = sum([1 for idx, row in df.iterrows() if os.path.exists(row['slowed_video_path'])])
print(f"Summary: Processed {num_processed} out of {num_videos} videos")
print(f"Slow videos saved to: {output_folder}")
print(f"Updated metadata saved to: {output_metadata_file}")

Reading metadata file...
Processing videos and updating metadata...


  0%|          | 0/281 [00:00<?, ?it/s]

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

Saving updated metadata to video_slow_metadata.csv...
Processing complete!
Summary: Processed 281 out of 281 videos
Slow videos saved to: videos_slow
Updated metadata saved to: video_slow_metadata.csv


# Even slower version

In [ ]:
import os
import re
import pandas as pd
import subprocess
from tqdm import tqdm
import csv
import shutil

# Function to slow down a video
def slow_down_video(input_path, output_path, speed_factor=0.5):
    """
    Slow down a video by the specified factor using ffmpeg.
    
    Args:
        input_path: Path to the input video
        output_path: Path to save the output video
        speed_factor: Factor to slow down the video (e.g., 0.5 for half speed)
    """
    # Create output directory if it doesn't exist
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    # Build ffmpeg command: use setpts filter to slow down video, atempo to slow down audio
    # atempo filter only supports 0.5-2.0 range, so for extreme slowing, we chain it
    cmd = [
        'ffmpeg', 
        '-i', input_path, 
        '-filter_complex', 
        f'[0:v]setpts={1/speed_factor}*PTS[v];[0:a]atempo={speed_factor}[a]', 
        '-map', '[v]', 
        '-map', '[a]', 
        '-c:v', 'libx264', 
        '-preset', 'medium', 
        '-c:a', 'aac', 
        output_path,
        '-y'  # Overwrite output file if it exists
    ]
    
    subprocess.run(cmd, check=True)

# Function to adjust SRT timestamps
def adjust_srt_timestamps(srt_text, speed_factor=0.5):
    """
    Adjust SRT timestamps by the specified speed factor.
    
    Args:
        srt_text: SRT format text
        speed_factor: Factor to slow down the timestamps (e.g., 0.5 for double duration)
    
    Returns:
        Adjusted SRT text
    """
    # Define a regex pattern to match SRT timestamp lines
    pattern = r'(\d+:\d+:\d+,\d+) --> (\d+:\d+:\d+,\d+)'
    
    def time_to_seconds(time_str):
        """Convert SRT timestamp to seconds"""
        h, m, s = time_str.replace(',', '.').split(':')
        return float(h) * 3600 + float(m) * 60 + float(s)
    
    def seconds_to_time(seconds):
        """Convert seconds to SRT timestamp format"""
        h = int(seconds // 3600)
        m = int((seconds % 3600) // 60)
        s = seconds % 60
        ms = int((s - int(s)) * 1000)
        s = int(s)
        return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"
    
    def adjust_timestamp(match):
        """Adjust a pair of timestamps based on the speed factor"""
        start_time = match.group(1)
        end_time = match.group(2)
        
        # Convert to seconds
        start_seconds = time_to_seconds(start_time)
        end_seconds = time_to_seconds(end_time)
        
        # Adjust time by dividing by speed factor (e.g., 0.5 makes duration 2x longer)
        adjusted_start = start_seconds / speed_factor
        adjusted_end = end_seconds / speed_factor
        
        # Convert back to timestamp format
        adjusted_start_time = seconds_to_time(adjusted_start)
        adjusted_end_time = seconds_to_time(adjusted_end)
        
        return f"{adjusted_start_time} --> {adjusted_end_time}"
    
    # Use regex to replace all timestamp lines
    adjusted_srt = re.sub(pattern, adjust_timestamp, srt_text)
    return adjusted_srt

# Main execution
SPEED_FACTOR = 0.5

# Create output folder
output_folder = "videos_slower"
os.makedirs(output_folder, exist_ok=True)

# Process metadata file
metadata_file = "video_slow_metadata.csv"
output_metadata_file = "video_slower_metadata.csv"

print("Reading metadata file...")
df = pd.read_csv(metadata_file)

# Create copies of dataframe columns to modify
df['slowed_video_path'] = df['video_path'].copy()
df['slowed_caption'] = df['caption'].copy()

# Process each video and update metadata
print("Processing videos and updating metadata...")
for idx, row in tqdm(df.iterrows(), total=len(df)):
    # Get video info
    video_path = row['video_path']
    
    # Check if video exists
    if not os.path.exists(video_path):
        print(f"Warning: Video not found: {video_path}")
        continue
    
    # Create output path
    output_path = video_path.replace("videos_slow/", "videos_slower/")
    
    # Slow down the video
    try:
        slow_down_video(video_path, output_path, speed_factor=SPEED_FACTOR)
        
        # Update metadata
        df.at[idx, 'slowed_video_path'] = output_path
        
        # Update caption timestamps if available
        if pd.notna(row['caption']):
            adjusted_caption = adjust_srt_timestamps(row['caption'], speed_factor=SPEED_FACTOR)
            df.at[idx, 'slowed_caption'] = adjusted_caption
    
    except Exception as e:
        print(f"Error processing {video_path}: {str(e)}")

# Create final output dataframe
output_df = df.copy()
output_df['video_path'] = df['slowed_video_path']  # Replace with slowed path
output_df['caption'] = df['slowed_caption']  # Replace with adjusted captions
output_df = output_df.drop(['slowed_video_path', 'slowed_caption'], axis=1)  # Remove temporary columns
# Double the duration of the video
output_df['duration'] = output_df['duration'] * 2

# Save updated metadata to CSV
print(f"Saving updated metadata to {output_metadata_file}...")
output_df.to_csv(output_metadata_file, index=False)

print("Processing complete!")

# Show a summary of the processing
num_videos = len(df)
num_processed = sum([1 for idx, row in df.iterrows() if os.path.exists(row['slowed_video_path'])])
print(f"Summary: Processed {num_processed} out of {num_videos} videos")
print(f"Slow videos saved to: {output_folder}")
print(f"Updated metadata saved to: {output_metadata_file}")

Reading metadata file...
Processing videos and updating metadata...


  0%|          | 0/281 [00:00<?, ?it/s]

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

Saving updated metadata to video_slower_metadata.csv...
Processing complete!
Summary: Processed 281 out of 281 videos
Slow videos saved to: videos_slower
Updated metadata saved to: video_slower_metadata.csv


# Full Video Set Version

In [4]:
import os
import re
import pandas as pd
import subprocess
from tqdm import tqdm
import csv
import shutil

# Function to slow down a video
def slow_down_video(input_path, output_path, speed_factor=0.5):
    """
    Slow down a video by the specified factor using ffmpeg.
    
    Args:
        input_path: Path to the input video
        output_path: Path to save the output video
        speed_factor: Factor to slow down the video (e.g., 0.5 for half speed)
    """
    # Create output directory if it doesn't exist
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    # Build ffmpeg command: use setpts filter to slow down video, atempo to slow down audio
    # atempo filter only supports 0.5-2.0 range, so for extreme slowing, we chain it
    cmd = [
        'ffmpeg', 
        '-i', input_path, 
        '-filter_complex', 
        f'[0:v]setpts={1/speed_factor}*PTS[v];[0:a]atempo={speed_factor}[a]', 
        '-map', '[v]', 
        '-map', '[a]', 
        '-c:v', 'libx264', 
        '-preset', 'medium', 
        '-c:a', 'aac', 
        output_path,
        '-y'  # Overwrite output file if it exists
    ]
    
    subprocess.run(cmd, check=True)

# Function to adjust SRT timestamps
def adjust_srt_timestamps(srt_text, speed_factor=0.5):
    """
    Adjust SRT timestamps by the specified speed factor.
    
    Args:
        srt_text: SRT format text
        speed_factor: Factor to slow down the timestamps (e.g., 0.5 for double duration)
    
    Returns:
        Adjusted SRT text
    """
    # Define a regex pattern to match SRT timestamp lines
    pattern = r'(\d+:\d+:\d+,\d+) --> (\d+:\d+:\d+,\d+)'
    
    def time_to_seconds(time_str):
        """Convert SRT timestamp to seconds"""
        h, m, s = time_str.replace(',', '.').split(':')
        return float(h) * 3600 + float(m) * 60 + float(s)
    
    def seconds_to_time(seconds):
        """Convert seconds to SRT timestamp format"""
        h = int(seconds // 3600)
        m = int((seconds % 3600) // 60)
        s = seconds % 60
        ms = int((s - int(s)) * 1000)
        s = int(s)
        return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"
    
    def adjust_timestamp(match):
        """Adjust a pair of timestamps based on the speed factor"""
        start_time = match.group(1)
        end_time = match.group(2)
        
        # Convert to seconds
        start_seconds = time_to_seconds(start_time)
        end_seconds = time_to_seconds(end_time)
        
        # Adjust time by dividing by speed factor (e.g., 0.5 makes duration 2x longer)
        adjusted_start = start_seconds / speed_factor
        adjusted_end = end_seconds / speed_factor
        
        # Convert back to timestamp format
        adjusted_start_time = seconds_to_time(adjusted_start)
        adjusted_end_time = seconds_to_time(adjusted_end)
        
        return f"{adjusted_start_time} --> {adjusted_end_time}"
    
    # Use regex to replace all timestamp lines
    adjusted_srt = re.sub(pattern, adjust_timestamp, srt_text)
    return adjusted_srt

# Main execution

# Create output folder
output_folder = "videos_full_slow"
os.makedirs(output_folder, exist_ok=True)

# Process metadata file
metadata_file = "video_full_metadata.csv"
output_metadata_file = "videos_full_slow_metadata.csv"

print("Reading metadata file...")
df = pd.read_csv(metadata_file)

# Create copies of dataframe columns to modify
df['slowed_video_path'] = df['video_path'].copy()
df['slowed_caption'] = df['caption'].copy()

# Process each video and update metadata
print("Processing videos and updating metadata...")
for idx, row in tqdm(df.iterrows(), total=len(df)):
    # Get video info
    video_path = row['video_path']
    
    # Check if video exists
    if not os.path.exists(video_path):
        print(f"Warning: Video not found: {video_path}")
        continue
    
    # Create output path
    output_path = video_path.replace("videos_full/", "videos_full_slow/")
    
    # Slow down the video
    try:
        slow_down_video(video_path, output_path)
        
        # Update metadata
        df.at[idx, 'slowed_video_path'] = output_path
        
        # Update caption timestamps if available
        if pd.notna(row['caption']):
            adjusted_caption = adjust_srt_timestamps(row['caption'])
            df.at[idx, 'slowed_caption'] = adjusted_caption
    
    except Exception as e:
        print(f"Error processing {video_path}: {str(e)}")

# Create final output dataframe
output_df = df.copy()
output_df['video_path'] = df['slowed_video_path']  # Replace with slowed path
output_df['caption'] = df['slowed_caption']  # Replace with adjusted captions
output_df = output_df.drop(['slowed_video_path', 'slowed_caption'], axis=1)  # Remove temporary columns
# Double the duration of the video, if duration value exists
output_df['duration'] = output_df['duration'] * 2


# Save updated metadata to CSV
print(f"Saving updated metadata to {output_metadata_file}...")
output_df.to_csv(output_metadata_file, index=False, encoding='utf-8-sig')

print("Processing complete!")

# Show a summary of the processing
num_videos = len(df)
num_processed = sum([1 for idx, row in df.iterrows() if os.path.exists(row['slowed_video_path'])])
print(f"Summary: Processed {num_processed} out of {num_videos} videos")
print(f"Slow videos saved to: {output_folder}")
print(f"Updated metadata saved to: {output_metadata_file}")

Reading metadata file...
Processing videos and updating metadata...


  0%|          | 0/292 [00:00<?, ?it/s]ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --

KeyboardInterrupt: 

In [5]:
# Create final output dataframe
output_df = df.copy()
output_df['video_path'] = df['slowed_video_path']  # Replace with slowed path
output_df['caption'] = df['slowed_caption']  # Replace with adjusted captions
output_df = output_df.drop(['slowed_video_path', 'slowed_caption'], axis=1)  # Remove temporary columns
# Double the duration of the video, if duration value exists
output_df['duration'] = output_df['duration'] * 2


# Save updated metadata to CSV
print(f"Saving updated metadata to {output_metadata_file}...")
output_df.to_csv(output_metadata_file, index=False, encoding='utf-8-sig')

print("Processing complete!")

# Show a summary of the processing
num_videos = len(df)
num_processed = sum([1 for idx, row in df.iterrows() if os.path.exists(row['slowed_video_path'])])
print(f"Summary: Processed {num_processed} out of {num_videos} videos")
print(f"Slow videos saved to: {output_folder}")
print(f"Updated metadata saved to: {output_metadata_file}")

Saving updated metadata to videos_full_slow_metadata.csv...
Processing complete!
Summary: Processed 291 out of 292 videos
Slow videos saved to: videos_full_slow
Updated metadata saved to: videos_full_slow_metadata.csv


In [6]:
import pandas as pd


# read videos_full_slow_metadata.csv
df = pd.read_csv("videos_full_slow_metadata.csv")

# in video_path column, replace videos_full  with videos_full_slow/
df['video_path'] = df['video_path'].str.replace("videos_full/", "videos_full_slow/")

# save the updated dataframe to a new csv file
df.to_csv("videos_full_slow_metadata.csv", index=False, encoding='utf-8-sig')

In [7]:
df

,youtube_url,error,title,description,caption,caption_language,publish_date,rating,channel_id,channel_url,thumbnail_url,channel_name,views,keywords,duration,video_path,video_id
0,https://www.youtube.com/shorts/-M7VdrQpWps,NaN,Spin Lightning,NaN,NaN,NaN,2024-07-16 07:20:51-07:00,NaN,UCOhFXcmfy8T11SgYFIrzY8Q,https://www.youtube.com/channel/UCOhFXcmfy8T11...,https://i.ytimg.com/vi/-M7VdrQpWps/sddefault.j...,Ideas 1+1,70431015.0,[],NaN,videos_full_slow/-M7VdrQpWps.mp4,-M7VdrQpWps.mp4
1,https://www.youtube.com/shorts/-Tf9VMl4krk,NaN,Monster Energy works better than Vinegar | Rus...,NaN,NaN,NaN,2024-08-19 14:47:17-07:00,NaN,UCK02oH_1vqL4VrUBEFXEUcA,https://www.youtube.com/channel/UCK02oH_1vqL4V...,https://i.ytimg.com/vi/-Tf9VMl4krk/sddefault.j...,Menard Metal Craft,20914321.0,"['Restoration', 'Science', 'ASMR', 'oddly sati...",NaN,videos_full_slow/-Tf9VMl4krk.mp4,-Tf9VMl4krk
2,https://www.youtube.com/shorts/-o7pVduTGVs,NaN,Now He’ll Never Leave😭,ill be over here with my broken back🫠,"1\r\n00:00:00,719 --> 00:00:04,600\r\nhey Mom ...",a.en,2024-05-23 18:54:30-07:00,NaN,UC1WbRuMMzbzUfBG65x1FmXQ,https://www.youtube.com/channel/UC1WbRuMMzbzUf...,https://i.ytimg.com/vi/-o7pVduTGVs/sddefault.j...,Peet Montzingo,320974272.0,"['peet montzingo', 'monzingo', 'mozingo', 'pet...",NaN,videos_full_slow/-o7pVduTGVs.mp4,-o7pVduTGVs
3,https://www.youtube.com/shorts/0-ugjds_Rz8,NaN,Stationary items that every student must have🦋...,NaN,"1\r\n00:00:00,000 --> 00:00:09,030\r\ncómo hac...",a.es,2022-06-30 04:51:47-07:00,NaN,UCIJ3_6Mby5AqCfGWM-XSk2w,https://www.youtube.com/channel/UCIJ3_6Mby5AqC...,https://i.ytimg.com/vi/0-ugjds_Rz8/sd2.jpg?sqp...,초코 민트 choco mint,1331558.0,[],33.34,videos_full_slow/0-ugjds_Rz8.mp4,0-ugjds_Rz8
4,https://www.youtube.com/shorts/0cKKHnAmncI,NaN,錯覺系列 箭頭方向 都一樣!? 視覺欺騙 有趣 好玩 科學 現象,利用視角與形狀的角度差來形成....\r\n\r\n喜歡我們系列的影片可以加LINE@ 搜尋...,NaN,NaN,2018-05-22 06:54:07-07:00,NaN,UClf9mmmtB6qdsPH-UNyAcnA,https://www.youtube.com/channel/UClf9mmmtB6qds...,https://i.ytimg.com/vi/0cKKHnAmncI/sd2.jpg?sqp...,JackWang,1168.0,"['視覺', '幻覺', '錯覺', '科學', '光學', '2018', '設計', '...",85.96,videos_full_slow/0cKKHnAmncI.mp4,0cKKHnAmncI
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
287,https://youtube.com/shorts/ly70cIZgrRA,NaN,This is why he owns the hotel,NaN,NaN,NaN,2024-10-08 10:01:38-07:00,NaN,UCC2lbKruS3-22iTD_1fe_MQ,https://www.youtube.com/channel/UCC2lbKruS3-22...,https://i.ytimg.com/vi/ly70cIZgrRA/sd2.jpg?sqp...,NXCRE,32959150.0,[],61.86,videos_full_slow/ly70cIZgrRA.mp4,ly70cIZgrRA
288,https://youtube.com/shorts/rCW1aGpFrJg,NaN,Limpando a escada rolante,NaN,"1\r\n00:00:05,090 --> 00:00:14,259\r\n[Music]\...",a.en,2024-01-27 16:33:52-08:00,NaN,UCtiddfzHXNevG-yYe5vc_4Q,https://www.youtube.com/channel/UCtiddfzHXNevG...,https://i.ytimg.com/vi/rCW1aGpFrJg/sd2.jpg?sqp...,Spider Slack,324516640.0,[],117.64,videos_full_slow/rCW1aGpFrJg.mp4,rCW1aGpFrJg
289,https://youtube.com/shorts/u2HvKEGq3Ik,u2HvKEGq3Ik is unavailable,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,videos_full_slow/u2HvKEGq3Ik.mp4,u2HvKEGq3Ik
290,https://youtube.com/shorts/vXUZpRRrIBU,NaN,Sour Family Sour Gummy Lemon Challenge! 🍋,#viral #gummy #challenge,"1\r\n00:00:00,120 --> 00:00:02,599\r\ntiny\r\n...",a.en,2024-04-27 22:00:33-07:00,NaN,UCm4FUt0uVITowfThQhs0NNQ,https://www.youtube.com/channel/UCm4FUt0uVITow...,https://i.ytimg.com/vi/vXUZpRRrIBU/sd2.jpg?sqp...,DrewMe2Jesus,125974312.0,"['sour king', 'sour king drew', 'sourking', 's...",60.64,videos_full_slow/vXUZpRRrIBU.mp4,vXUZpRRrIBU
